# Module 05 — Extended Thinking

Enable a **private reasoning budget** so **`claude-opus-4-8`** reasons step-by-step before answering — useful for hard math, logic puzzles, and multi-step planning.

This builds on **Modules 00–04** and uses the same **`AnthropicVertex` / ADC** path.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]"

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Enable thinking

Turn on extended thinking with the **`thinking`** parameter:

```python
thinking={"type": "enabled", "budget_tokens": 1024}
```

Key constraints:

- **`budget_tokens` minimum is 1024.**
- **`max_tokens` must be greater than `budget_tokens`** — here `max_tokens=2048`, `budget_tokens=1024`.
- **Leave `temperature` / `top_p` at their defaults** (don't set them) — the API restricts sampling params while thinking is on.

The response `content` then contains **thinking block(s)** (`block.type == "thinking"`, with `block.thinking` text + a signature) **plus text block(s)** (`block.type == "text"`).

In [ ]:
resp = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    thinking={"type": "enabled", "budget_tokens": 1024},
    messages=[{
        "role": "user",
        "content": (
            "A train leaves at 9:00 AM traveling 60 km/h. A second train leaves the "
            "same station at 10:00 AM on the same track traveling 90 km/h in the same "
            "direction. At what time does the second train catch up to the first?"
        ),
    }],
)

for block in resp.content:
    if block.type == "thinking":
        print("===== THINKING =====")
        print(block.thinking)
    elif block.type == "text":
        print("\n===== ANSWER =====")
        print(block.text)

## Part D — Inspect thinking + usage

The **thinking block** carries the reasoning (plus a cryptographic **signature**); the **text block** is the final answer. Thinking tokens are billed as **output tokens**, so they count toward `usage.output_tokens`.

In [ ]:
thinking_text = next((b.thinking for b in resp.content if b.type == "thinking"), "")
answer_text = next((b.text for b in resp.content if b.type == "text"), "")

print("Thinking snippet:", thinking_text[:200], "...")
print("\nFinal answer:", answer_text)
print("-" * 60)
print(f"input_tokens : {resp.usage.input_tokens}")
print(f"output_tokens: {resp.usage.output_tokens}  (includes thinking tokens)")

## Part E — Streaming thinking (incremental)

Streaming surfaces **both the thinking and the answer incrementally** via stream events — useful for showing live progress on long reasoning. Watch for `content_block_delta` events and branch on the delta type (`thinking_delta` vs `text_delta`).

In [ ]:
printed_thinking = False
printed_answer = False

with client.messages.stream(
    model=MODEL,
    max_tokens=2048,
    thinking={"type": "enabled", "budget_tokens": 1024},
    messages=[{"role": "user", "content": "What is 17 * 24, and is the result divisible by 3? Show your reasoning."}],
) as stream:
    for event in stream:
        if event.type == "content_block_delta":
            if event.delta.type == "thinking_delta":
                if not printed_thinking:
                    print("\n[THINKING] ", end="")
                    printed_thinking = True
                print(event.delta.thinking, end="")
            elif event.delta.type == "text_delta":
                if not printed_answer:
                    print("\n\n[ANSWER] ", end="")
                    printed_answer = True
                print(event.delta.text, end="")

## Part F — Notes & recap

### Notes

- **When to use:** hard reasoning, multi-step math, logic, and planning. For simple tasks it adds latency/cost with little benefit.
- **Budget:** start around **1024** tokens and raise for harder problems. **`max_tokens` must exceed `budget_tokens`.**
- **Sampling params:** keep `temperature` / `top_p` at their defaults while thinking is enabled.
- **With tool use across turns:** you must **preserve the thinking blocks** in the message history (send them back unchanged) for multi-turn tool-use flows.
- See the Anthropic **extended thinking** docs (https://docs.claude.com/en/docs/build-with-claude/extended-thinking) and confirm the current details there, as they may change.

### Recap

- Enable with `thinking={"type": "enabled", "budget_tokens": 1024}`; `budget_tokens` min is 1024 and `max_tokens` must be larger.
- Leave `temperature` / `top_p` default while thinking is on.
- The response holds **thinking block(s)** (reasoning + signature) and **text block(s)** (the answer); thinking is billed as output tokens.
- You can **stream** thinking and the answer incrementally via `content_block_delta` events.

**Next: Module 06 — Web search.**